In [1]:
%%capture
!pip install -U lightautoml

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import requests
import joblib

import numpy
import pandas
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import AdaBoostRegressor

In [3]:
import torch
from lightautoml.automl.presets.tabular_presets import TabularAutoML, TabularUtilizedAutoML
from lightautoml.tasks import Task
from lightautoml.report.report_deco import ReportDeco

In [4]:
N_THREADS = 4
N_FOLDS = 8
RANDOM_STATE = 42
TEST_SIZE = 0.2
TIMEOUT = 3600*100

numpy.random.seed(RANDOM_STATE)
torch.set_num_threads(N_THREADS)

In [5]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['efs_time','ID'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [6]:
test.columns, train.columns

(Index(['dri_score', 'psych_disturb', 'cyto_score', 'diabetes',
        'hla_match_c_high', 'hla_high_res_8', 'tbi_status', 'arrhythmia',
        'hla_low_res_6', 'graft_type', 'vent_hist', 'renal_issue',
        'pulm_severe', 'prim_disease_hct', 'hla_high_res_6', 'cmv_status',
        'hla_high_res_10', 'hla_match_dqb1_high', 'tce_imm_match', 'hla_nmdp_6',
        'hla_match_c_low', 'rituximab', 'hla_match_drb1_low',
        'hla_match_dqb1_low', 'prod_type', 'cyto_score_detail',
        'conditioning_intensity', 'ethnicity', 'year_hct', 'obesity', 'mrd_hct',
        'in_vivo_tcd', 'tce_match', 'hla_match_a_high', 'hepatic_severe',
        'donor_age', 'prior_tumor', 'hla_match_b_low', 'peptic_ulcer',
        'age_at_hct', 'hla_match_a_low', 'gvhd_proph', 'rheum_issue',
        'sex_match', 'hla_match_b_high', 'race_group', 'comorbidity_score',
        'karnofsky_score', 'hepatic_mild', 'tce_div_match', 'donor_related',
        'melphalan_dose', 'hla_low_res_8', 'cardiac', 'hla_match

In [7]:
task = Task('reg')

lgb_params = {
    'metric': 'RMSE',
    'lambda_l1': 1e-07, 
    'lambda_l2': 2e-07, 
    'num_leaves': 42, 
    'feature_fraction': 0.55, 
    'bagging_fraction': 0.9, 
    'bagging_freq': 3, 
    'min_child_samples': 19,
    'num_threads': 4
}

cb_params = {
    'num_trees': 10000, 
    'od_wait': 1200, 
    'learning_rate': 0.2, 
    'l2_leaf_reg': 64, 
    'subsample': 0.83, 
    'random_strength': 17.17,  
    'min_data_in_leaf': 10, 
    'leaf_estimation_iterations': 3,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'bootstrap_type': 'Bernoulli',
    'leaf_estimation_method': 'Newton',
    'random_seed': 42,
    "thread_count": 4
}

In [8]:
automl = TabularAutoML(task = task, 
                       timeout = TIMEOUT,
                       cpu_limit = N_THREADS,
                       reader_params = {'n_jobs': N_THREADS, 'cv': N_FOLDS, 'random_state': RANDOM_STATE},
                       general_params = {'use_algos': [['lgb', 'cb', "lgb_tuned"]]}, # LGBM and CatBoost algos only
                       lgb_params = {'default_params': lgb_params, 'freeze_defaults': True}, # LGBM params
                       cb_params = {'default_params': cb_params, 'freeze_defaults': True}, # CatBoost params
                      )

RD = ReportDeco(output_path = 'tabularAutoML_model_report')
automl_rd = RD(automl)

out_of_fold_predictions = automl_rd.fit_predict(
    train,
    roles = {
        'target': 'efs',
        #'drop': ['unique_id']
        #'weights': 'weight'
    }, 
    verbose = 2
)

[04:25:30] Stdout logging level is INFO2.
[04:25:30] Copying TaskTimer may affect the parent PipelineTimer, so copy will create new unlimited TaskTimer
[04:25:30] Task: reg

[04:25:30] Start automl preset with listed constraints:
[04:25:30] - time: 360000.00 seconds
[04:25:30] - CPU: 4 cores
[04:25:30] - memory: 16 GB

[04:25:30] Train data shape: (28800, 58)

[04:25:39] Layer 1 train process start. Time left 359991.80 secs
[04:25:43] Selector_LightGBM fitting and predicting completed
[04:25:45] Start fitting Lvl_0_Pipe_0_Mod_0_LightGBM ...
[04:25:45] ===== Start working with fold 0 for Lvl_0_Pipe_0_Mod_0_LightGBM =====
[04:25:48] ===== Start working with fold 1 for Lvl_0_Pipe_0_Mod_0_LightGBM =====
[04:25:50] ===== Start working with fold 2 for Lvl_0_Pipe_0_Mod_0_LightGBM =====
[04:25:54] ===== Start working with fold 3 for Lvl_0_Pipe_0_Mod_0_LightGBM =====
[04:25:58] ===== Start working with fold 4 for Lvl_0_Pipe_0_Mod_0_LightGBM =====
[04:26:00] ===== Start working with fold 5 for L

In [9]:
joblib.dump(automl, 'model.pkl')

['model.pkl']

automl = joblib.load('/kaggle/input/cibmtr-lightautoml-model/model.pkl')

In [10]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']
test_predictions = automl.predict(test)

In [11]:
submission = pandas.DataFrame({
    'id': id.values,
    'sales_hat': test_predictions.data.reshape(-1),
})
submission

,id,sales_hat
0,28800,0.134496
1,28801,0.692168
2,28802,-0.009174


In [12]:
submission.to_csv('submission.csv', index = False)